In [1]:
import os, json
import pandas as pd
import openai
from dotenv import load_dotenv
from pathlib import Path
from tqdm import tqdm
import time

In [2]:
# Load API
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")
MODEL_NAME     = "gpt-4o-mini-2024-07-18" 
TEMPERATURE    = 1.0 # Default
print("API Key loaded:", openai.api_key is not None)

# Load files
src_file  = "../data/disaster_description.csv"
out_file  = "extracted_onestage.csv"
df = pd.read_csv(src_file)
# df = df.head(200)  # For test

API Key loaded: True


# Materials

In [4]:
DISASTER_CLASSIFICATION = json.load(open("../data/classification_document.json"))

In [5]:
MAGNITUDE_INFORMATION = json.load(open("../data/magnitude_document.json"))

In [6]:
FIELD_SCHEMA = json.load(open("../data/schema_document.json"))

In [7]:
CATEGORICAL_VALUES = {
    "Disaster Classification": DISASTER_CLASSIFICATION,
    "Magnitude": MAGNITUDE_INFORMATION
}

# Prompt

In [9]:
OUTPUT_FORMAT = {
  "Event": {"fields": "values"},
  "Geographical Information": {"fields": "values"},
  "Effect": {"fields": "values"}
}

In [10]:
prompt = f"""
        Please refer FIELD_SCHEMA first, then extract all fields that are marked as 'Required' or are mentioned in DISASTER_DESCRIPTION,
        recording each value exactly (using only the categorical options in CATEGORICAL_VALUES). 
        Finally return a single JSON object that conforms precisely to OUTPUT_FORMAT.
        
        FIELD_SCHEMA:
        {json.dumps(FIELD_SCHEMA, ensure_ascii=False)}
        
        CATEGORICAL_VALUES:
        {json.dumps(CATEGORICAL_VALUES, ensure_ascii=False)}
        
        OUTPUT_FORMAT:
        {json.dumps(OUTPUT_FORMAT, ensure_ascii=False)}
    """

# Implement

In [12]:
def call_llm(prompt: str, description: str) -> dict:
    try:
        resp = openai.chat.completions.create(
            model           = MODEL_NAME,
            messages        = [{"role": "system", "content": prompt},
                               {"role":"user", "content": description}],
            temperature     = TEMPERATURE,
            response_format = {"type": "json_object"} # Limit output in JSON format
        )
        return json.loads(resp.choices[0].message.content)
    except Exception as e:
        print("LLM Error:", e)
        return None

In [13]:
results = []
for desc in tqdm(df["description"], desc="one extracting"):
    extracted = call_llm(prompt,desc)
    results.append(extracted)
    time.sleep(0.5)

df["onestage_extracted"] = results
df.to_csv(out_file, index=False)

one extracting:  21%|████▉                   | 327/1588 [40:37<52:26,  2.49s/it]

LLM Error: Out of range float values are not JSON compliant


one extracting:  23%|█████▍                  | 359/1588 [42:00<54:48,  2.68s/it]

LLM Error: Out of range float values are not JSON compliant


one extracting: 100%|█████████████████████| 1588/1588 [5:56:12<00:00, 13.46s/it]
